# 矩阵分解 (Matrix Factorization) 实战

本 Notebook 对应文档 `05_矩阵分解_SVD.md`，旨在通过代码实践深入理解矩阵分解算法。

我们将包含两个部分：
1. **原理实现**：使用 NumPy 手写一个 `BiasSVD` 类，展示其内部的梯度下降更新逻辑。
2. **工程实现**：介绍在实际工业界场景中，如何高效实现矩阵分解（如使用 ALS 并行计算或深度学习框架）。

## 1. 原理实现：手写 BiasSVD

为了理解 MF 的核心原理，我们实现一个带有偏置项（Bias）的矩阵分解模型。该模型的目标函数为：

$$
\hat{r}_{ui} = \mu + b_u + b_i + p_u \cdot q_i
$$

In [17]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin

class BiasSVD(BaseEstimator, RegressorMixin):
    """
    BiasSVD 矩阵分解算法实现 (教学演示版)
    """
    def __init__(self, n_factors=20, n_epochs=20, lr=0.01, reg=0.02, random_state=42):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def fit(self, X, y):
        """
        X: shape (n_samples, 2), columns: [user_id, item_id]
        y: shape (n_samples, ), ratings
        """
        np.random.seed(self.random_state)

        # 1. ID 映射 (Mapping): 将原始 ID 转为 0 ~ N-1 的矩阵索引
        self.user_map = {u: i for i, u in enumerate(np.unique(X[:, 0]))}
        self.item_map = {i: j for j, i in enumerate(np.unique(X[:, 1]))}

        n_users, n_items = len(self.user_map), len(self.item_map)
        print(f"Training with {n_users} users and {n_items} items...")

        # 2. 参数初始化 (Initialization)
        self.mu = np.mean(y)                   # 全局平均分
        self.bu = np.zeros(n_users)            # 用户偏置
        self.bi = np.zeros(n_items)            # 物品偏置
        self.P = np.random.normal(0, 0.1, (n_users, self.n_factors)) # 用户隐矩阵
        self.Q = np.random.normal(0, 0.1, (n_items, self.n_factors)) # 物品隐矩阵

        # 3. SGD 训练 (Stochastic Gradient Descent)
        for epoch in range(self.n_epochs):
            total_loss = 0
            for k in range(len(y)):
                u_orig, i_orig = X[k]
                r = y[k]

                u, i = self.user_map[u_orig], self.item_map[i_orig]

                # [核心逻辑] 前向传播：计算预测值
                # Formula: \hat{r}_{ui} = \mu + b_u + b_i + p_u \cdot q_i
                dot = np.dot(self.P[u], self.Q[i])
                pred = self.mu + self.bu[u] + self.bi[i] + dot

                # 计算误差
                err = r - pred

                # [核心逻辑] 反向传播：更新参数
                # Formula: b_u <- b_u + \eta * (err - \lambda * b_u)
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (err - self.reg * self.bi[i])

                # 更新隐向量 P 和 Q
                pu_old = self.P[u].copy() # 暂存旧值
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * pu_old - self.reg * self.Q[i])

                total_loss += err**2

            if (epoch + 1) % 5 == 0:
                mse = total_loss / len(y)
                print(f"Epoch {epoch+1}/{self.n_epochs} - MSE: {mse:.4f}")

        return self

    def predict(self, X):
        preds = []
        for u_orig, i_orig in X:
            # 处理冷启动：如果是新用户或新物品，返回全局平均分
            if u_orig not in self.user_map or i_orig not in self.item_map:
                preds.append(self.mu)
                continue

            u = self.user_map[u_orig]
            i = self.item_map[i_orig]

            pred = self.mu + self.bu[u] + self.bi[i] + np.dot(self.P[u], self.Q[i])
            preds.append(pred)
        return np.array(preds)

### 1.1 构造测试数据并运行

我们使用文档中 2.1 节提到的简单案例数据进行测试。

In [18]:
# 构造数据
# User-Item-Rating
X_train = np.array([
    ['U1', 'M1'], ['U1', 'M3'],
    ['U2', 'M1'], ['U2', 'M4'],
    ['U3', 'M2'], ['U3', 'M4'],
    ['U4', 'M1'], ['U4', 'M3']
])
y_train = np.array([5, 3, 4, 2, 4, 5, 3, 4])

# 实例化并训练模型
svd = BiasSVD(n_factors=2, n_epochs=50, lr=0.01, reg=0.02)
svd.fit(X_train, y_train)

# 预测一个缺失值: U1 对 M2 (U1 喜欢动作片, M2 是爱情片)
X_test = np.array([['U1', 'M2']])
pred = svd.predict(X_test)
print(f"\n预测 U1 对 M2 的评分: {pred[0]:.2f}")

# 打印隐向量查看
print("\nU1 的隐向量:", svd.P[svd.user_map['U1']])
print("M1 (动作片) 的隐向量:", svd.Q[svd.item_map['M1']])

Training with 4 users and 4 items...
Epoch 5/50 - MSE: 0.8798
Epoch 10/50 - MSE: 0.8160
Epoch 15/50 - MSE: 0.7622
Epoch 20/50 - MSE: 0.7165
Epoch 25/50 - MSE: 0.6772
Epoch 30/50 - MSE: 0.6433
Epoch 35/50 - MSE: 0.6135
Epoch 40/50 - MSE: 0.5872
Epoch 45/50 - MSE: 0.5635
Epoch 50/50 - MSE: 0.5421

预测 U1 对 M2 的评分: 3.86

U1 的隐向量: [0.00693418 0.08614825]
M1 (动作片) 的隐向量: [-0.07358519  0.08512566]


---

## 2. 工程实现说明

在上述的原理实现中，我们使用 Python 循环进行 SGD 更新。这种方式直观但效率较低，无法处理大规模数据集。在工业界实际工程中，我们通常采用以下两种更高效的方案，以避免手动编写低效循环并利用硬件加速。

### 2.1 方案一：ALS (交替最小二乘法)

**原理**：
ALS (Alternating Least Squares) 是一种非梯度下降的优化方法。它的核心思想是：固定用户矩阵 $P$，那么求解物品矩阵 $Q$ 就变成了一个标准的线性回归（最小二乘）问题；反之亦然。通过交替固定一方优化另一方，可以将非凸问题转化为一系列凸优化问题。

**工程优势**：
- **并行化**：每次固定 $P$ 求解 $Q$ 时，每个物品的求解是相互独立的，因此可以极易地并行化（例如 Spark MLlib 的实现）。
- **数学优化**：利用 BLAS/LAPACK 库进行矩阵运算，速度远快于标量循环。

**常用库**：
- **Apache Spark MLlib**：大数据场景下的标准选择。
- **implicit**：专门针对隐式反馈数据的高性能 Python 库。

### 2.2 方案二：基于 Embedding 的深度学习模型

**原理**：
将矩阵分解看作是一个简单的神经网络：用户和物品 ID 输入到 Embedding 层，得到的向量进行点积（Dot Product），即得到预测评分。

**工程优势**：
- **GPU 加速**：利用 PyTorch/TensorFlow 等框架，天然支持 GPU 训练。
- **模型扩展**：非常容易加入其他特征（如用户年龄、物品分类）或升级为复杂的网络结构（如 NeuralCF, DeepFM）。

**注意事项**：
在轻量级环境或仅依赖 CPU 的场景下，引入 PyTorch 等重型框架可能会显著增加部署体积（Docker Image Size）。此时，ALS 或基于 C++ 扩展的轻量级库是更好的选择。